# 03 — Microstructure Features

Visualize each feature from Phase 5 over a week of synthetic BTCUSDT data.
Flag any NaNs or outliers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.random.seed(42)

# Generate 1 week of 1-minute synthetic BTCUSDT data
n = 7 * 24 * 60  # 10080 bars
rng = np.random.default_rng(42)

close = 30000.0 + np.cumsum(rng.normal(0, 10, n))
close = np.maximum(close, 20000.0)
high = close + rng.uniform(5, 50, n)
low = close - rng.uniform(5, 50, n)
open_ = close + rng.normal(0, 15, n)
volume = rng.uniform(10, 500, n)

bid_price = close - rng.uniform(0.5, 5, n)
ask_price = close + rng.uniform(0.5, 5, n)
bid_size = rng.uniform(0.1, 10, n)
ask_size = rng.uniform(0.1, 10, n)

funding = np.where(rng.random(n) > 0.875, rng.normal(0.0001, 0.0005, n), np.nan)
spot_price = close - rng.uniform(0, 10, n)
btc_return = np.log(close / np.roll(close, 1))
btc_return[0] = 0

df = pd.DataFrame(
    {
        "event_time": pd.date_range("2023-06-01", periods=n, freq="1min"),
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
        "bid_price": bid_price,
        "ask_price": ask_price,
        "bid_size": bid_size,
        "ask_size": ask_size,
        "funding_rate": funding,
        "spot_price": spot_price,
        "btc_return": btc_return,
    }
)
print(f"DataFrame shape: {df.shape}")
df.head()

In [ ]:
from tessera.features import (
    VPIN,
    BetaToBTC,
    FeaturePipeline,
    FundingRate,
    FundingZScore,
    GarmanKlass,
    IdiosyncraticResidual,
    LogReturn,
    MicroPrice,
    OrderFlowImbalance,
    Parkinson,
    RealizedVol,
    SpotPerpBasis,
    SpreadBps,
    UniverseRank,
    VolOfVol,
)

features = [
    LogReturn(horizon=1),
    LogReturn(horizon=5),
    LogReturn(horizon=60),
    RealizedVol(window=300),
    RealizedVol(window=60),
    Parkinson(window=60),
    GarmanKlass(window=60),
    VolOfVol(window=60),
    OrderFlowImbalance(depth=1),
    MicroPrice(),
    SpreadBps(),
    VPIN(bucket_size=1000, window=50),
    FundingRate(),
    FundingZScore(window=720),
    SpotPerpBasis(),
    BetaToBTC(window=1440),
    IdiosyncraticResidual(window=1440),
    UniverseRank(metric="log_return_1"),
]

pipeline = FeaturePipeline(features, cache_dir=None)
result = pipeline.compute(df, symbol="BTCUSDT", use_cache=False)
print(f"Result shape: {result.shape}")
print(f"Feature columns: {[f.name for f in features]}")

In [ ]:
# NaN report
feature_names = [f.name for f in features]
nan_report = result[feature_names].isna().sum()
nan_pct = nan_report / len(result) * 100
print("=== NaN Report ===")
for name in feature_names:
    pct = nan_pct[name]
    flag = " ⚠️ HIGH" if pct > 50 else ""
    print(f"  {name:30s}: {nan_report[name]:5d} NaNs ({pct:.1f}%){flag}")

In [ ]:
# Visualize each feature
fig, axes = plt.subplots(len(feature_names), 1, figsize=(14, 3 * len(feature_names)), sharex=True)

for ax, name in zip(axes, feature_names, strict=False):
    series = result[name].dropna()
    ax.plot(result["event_time"].iloc[series.index], series.values, linewidth=0.5)
    ax.set_ylabel(name, fontsize=8)
    ax.grid(True, alpha=0.3)

    # Flag outliers (>4 std)
    if len(series) > 0:
        mean, std = series.mean(), series.std()
        if std > 0:
            outliers = series[abs(series - mean) > 4 * std]
            if len(outliers) > 0:
                ax.scatter(
                    result["event_time"].iloc[outliers.index],
                    outliers.values,
                    c="red",
                    s=10,
                    zorder=5,
                    label=f"{len(outliers)} outliers",
                )
                ax.legend(fontsize=7)

axes[-1].set_xlabel("Time")
plt.suptitle("Phase 5 Features — 1 Week BTCUSDT (Synthetic)", y=1.0)
plt.tight_layout()
plt.savefig("../docs/figures/features_visualization.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# Correlation matrix of all features
corr = result[feature_names].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(feature_names)))
ax.set_yticks(range(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha="right", fontsize=7)
ax.set_yticklabels(feature_names, fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../docs/figures/feature_correlation_matrix.png", dpi=100, bbox_inches="tight")
plt.show()

print("\n=== High correlations (|r| > 0.8) ===")
for i in range(len(feature_names)):
    for j in range(i + 1, len(feature_names)):
        r = corr.iloc[i, j]
        if abs(r) > 0.8:
            print(f"  {feature_names[i]} ↔ {feature_names[j]}: {r:.3f}")

In [ ]:
# Summary statistics
print("=== Feature Summary Statistics ===")
result[feature_names].describe().T